# Monte Carlo Noise Validation

A deterministic cubesim result predicts the target signal and detector variance for each registered aperture. Monte Carlo realizations drawn from the same result test whether its configured Poisson and Gaussian detector statistics reproduce those predictions.

In [ ]:
import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np

import cubesim
import inclined_galaxy

## Calculation

`cubesim.Etc` loads the instrument definition, and `inclined_galaxy.configure()` adds the inclined emission-line galaxy, exposure sequence, and three registered apertures. `Etc.run()` returns the expected detector signals, variances, and aperture reductions needed for sampling.

In [ ]:
etc = cubesim.Etc("instrument_data")
inclined_galaxy.configure(etc)
result = etc.run()

## Cube Sampling

`EtcResult.sample()` draws a sky-subtracted detector realization from the expected signals and detector noise. The returned object retains the sampled data, wavelength coordinate, calculation options, and random seed.

In [ ]:
sampled_cube = result.sample(seed=42)

## Aperture Sampling

`ApertureResult.sample()` draws integrated measurements using the exposure and sky-subtraction configuration retained by the result. Each returned object retains its sampled data, aperture definition, and random seed. Subtracting `aperture.signals.target` from `sample.data` leaves residuals around the expected target measurement.

In [ ]:
sample_count = 1000
diagnostics = []

for index, aperture in enumerate(result.apertures):
    sample = aperture.sample(n=sample_count, seed=100 + index)
    expected_signal = aperture.signals.target
    predicted_variance = aperture.variances.total

    residuals = sample.data - expected_signal
    measured_variance = residuals.var(ddof=1)
    variance_ratio = (measured_variance / predicted_variance).to_value(
        u.dimensionless_unscaled
    )
    standardized_residuals = (
        residuals / np.sqrt(predicted_variance)
    ).to_value(
        u.dimensionless_unscaled
    )

    diagnostics.append(
        {
            "name": aperture.name,
            "predicted_variance": predicted_variance,
            "measured_variance": measured_variance,
            "variance_ratio": variance_ratio,
            "standardized_residuals": standardized_residuals,
        }
    )

## Variance Comparison

The measured-to-predicted variance ratio compares the sampled noise with `aperture.variances.total`. A ratio near one indicates agreement between the realizations and the detector-noise prediction.

In [ ]:
for diagnostic in diagnostics:
    print(
        f"{diagnostic['name']:11s}  "
        f"predicted={diagnostic['predicted_variance'].value:12.3f} electron2  "
        f"measured={diagnostic['measured_variance'].value:12.3f} electron2  "
        f"ratio={diagnostic['variance_ratio']:.3f}"
    )

## Noise Distributions

Dividing each residual by the predicted standard deviation places all apertures on a common scale with expected mean zero and variance one. The histograms show the sampled distributions; the dashed normal curves show the high-count Gaussian approximation to the Poisson-plus-Gaussian noise.

In [ ]:
figure, axes = plt.subplots(
    len(diagnostics),
    1,
    figsize=(7, 3 * len(diagnostics)),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)
bins = np.linspace(-4, 4, 41)
expected_x = np.linspace(-4, 4, 500)
expected_density = np.exp(-0.5 * expected_x**2) / np.sqrt(2 * np.pi)
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"][: len(diagnostics)]
for axis, diagnostic, color in zip(axes, diagnostics, colors, strict=True):
    name = diagnostic["name"]
    values = diagnostic["standardized_residuals"]
    variance_ratio = diagnostic["variance_ratio"]
    axis.hist(
        values,
        bins=bins,
        density=True,
        histtype="step",
        linewidth=1.5,
        color=color,
        label="Samples",
    )
    axis.plot(
        expected_x,
        expected_density,
        color="black",
        linestyle="--",
        linewidth=1.5,
        label="Gaussian expectation",
    )
    axis.set(
        xlim=(-4, 4),
        ylabel="Probability density",
        title=f"{name}: variance ratio = {variance_ratio:.3f}",
    )
    axis.grid(alpha=0.2)
    axis.legend()
axes[-1].set_xlabel("Noise / predicted standard deviation")
figure.suptitle(f"Aperture noise from {sample_count:,} realizations")
plt.show()

## Save Sampled Cube

`SampledCube.save()` writes the realization to a FITS file with its wavelength coordinate, units, calculation metadata, random seed, and cube coordinates. `overwrite=True` allows the sampling calculation to be rerun using the same output path.

In [ ]:
sampled_cube.save("sampled_cube.fits", overwrite=True)